#### Training Using Pytorch

#### 1. Manual training

In [46]:
import torch


In [47]:
x = torch.tensor([
    [1.0,2.0,3.0,4.0,5.0,6.0],
    [50.0,55.0,60.0,65.0,70.0,75.0],
])

y = torch.tensor([[0, 0, 0, 1, 1, 1]], dtype=torch.float32)


In [48]:
X_mean = x.mean(dim=1,keepdim=True)
X_range = x.max(dim=1,keepdim=True).values - x.min(dim=1,keepdim=True).values

X_norm = (x - X_mean) / X_range


##### Defining the model architecture

In [ ]:
class LogisticPerceptron():

    def __init__(self,X):
        self.weights=torch.randn(X.shape[0],1,requires_grad=True)
        self.bias=torch.zeros(1,requires_grad=True)
    
    def forward_prop(self,X):
        z=torch.matmul(self.weights.T,X)+self.bias
        a=torch.sigmoid(z)
        
        return a
    

    def binary_cross_entropy(self,a,y):
        loss = -(y*torch.log(a+1e-8) + (1-y)*torch.log(1-a+1e-8)).mean()

        return loss

    

    

In [50]:
lr=0.01
epochs=10

In [51]:
model=LogisticPerceptron(X_norm)

In [52]:
model.weights

tensor([[1.0889],
        [2.8892]], requires_grad=True)

##### Training

In [53]:
for epoch in range(epochs):
    #forward prop

    y_pred=model.forward_prop(X_norm)

    #loss

    loss=model.binary_cross_entropy(y_pred,y)

    print(f"epoch : {epoch+1} || loss : {loss.item():.4f}")

    # Backpropagation

    loss.backward()


    # updating parameters

    with torch.no_grad():
        model.weights-= lr*model.weights.grad
        model.bias-=lr*model.bias.grad
    

    # empty the gradient
    model.weights.grad.zero_()
    model.bias.grad.zero_()






    

epoch : 1 || loss : 0.3023
epoch : 2 || loss : 0.3023
epoch : 3 || loss : 0.3022
epoch : 4 || loss : 0.3021
epoch : 5 || loss : 0.3021
epoch : 6 || loss : 0.3020
epoch : 7 || loss : 0.3019
epoch : 8 || loss : 0.3019
epoch : 9 || loss : 0.3018
epoch : 10 || loss : 0.3017


##### Prediction

In [57]:
test_hours = torch.tensor([[4.5],[68]])
test_norm = (test_hours - X_mean) / X_range

with torch.no_grad():

  prob = model.forward_prop(test_norm)
  prediction = (prob >= 0.5).float()

print("\nProbability of passing:", prob.item())
print("Prediction (0=Fail, 1=Pass):", prediction.item())


Probability of passing: 0.7017781138420105
Prediction (0=Fail, 1=Pass): 1.0


##### 2. The NN  and the Optim Module

##### 1 sigmoid-Perceptron

In [1]:
import torch

In [2]:
import torch.nn as nn

In [3]:
X = torch.tensor([
    [1, 50],
    [2, 55],
    [3, 60],
    [4, 65],
    [5, 70],
    [6, 75]
], dtype=torch.float32)

Y = torch.tensor([[0], [0], [0], [1], [1], [1]],dtype=torch.float32)

In [4]:
X_mean = X.mean(dim=0,keepdim=True)
X_range = X.max(dim=0,keepdim=True).values - X.min(dim=0,keepdim=True).values

X2_norm = (X - X_mean) / X_range


In [5]:
X2_norm.shape

torch.Size([6, 2])

In [6]:
class Model2(nn.Module):
    
    def __init__(self,features):

        super().__init__()
        self.linear=nn.Linear(features,1)
        self.sigmoid=nn.Sigmoid()

    def forward(self,x):
        out=self.linear(x)
        out=self.sigmoid(out)
        return out


In [ ]:
model2=Model2(X2_norm.shape[1])
model2(X2_norm)

tensor([[0.6200],
        [0.5739],
        [0.5263],
        [0.4784],
        [0.4308],
        [0.3844]], grad_fn=<SigmoidBackward0>)

In [9]:
model2.linear.weight

Parameter containing:
tensor([[0.6726, 0.6933]], requires_grad=True)

In [10]:
model2.linear.bias

Parameter containing:
tensor([-0.0112], requires_grad=True)

In [11]:
! pip install torchinfo

In [12]:
from torchinfo import summary
summary(model2,input_size=(6,2))


Layer (type:depth-idx)                   Output Shape              Param #
Model2                                   [6, 1]                    --
├─Linear: 1-1                            [6, 1]                    3
├─Sigmoid: 1-2                           [6, 1]                    --
Total params: 3
Trainable params: 3
Non-trainable params: 0
Total mult-adds (M): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

##### 2. MlP

##### Model Architecture

In [13]:
class Model3(nn.Module):
    
    def __init__(self,features):

        super().__init__()
        self.linear1=nn.Linear(features,3)
        self.relu=nn.ReLU()
        self.linear2=nn.Linear(3,1)
        self.sigmoid=nn.Sigmoid()

    def forward(self,x):
        out=self.linear1(x)
        out=self.relu(out)
        out=self.linear2(out)
        out=self.sigmoid(out)
        return out

In [14]:
 def binary_cross_entropy(a,y):
        loss = -(y*torch.log(a+1e-8) + (1-y)*torch.log(1-a+1e-8)).mean()

        return loss

In [15]:
loss_function=nn.BCELoss()

In [16]:
model3= Model3(X2_norm.shape[1])
model3(X2_norm)

tensor([[0.5669],
        [0.5847],
        [0.6024],
        [0.6328],
        [0.6624],
        [0.6909]], grad_fn=<SigmoidBackward0>)

In [17]:
from torchinfo import summary
summary(model3,input_size=(6,2))

Layer (type:depth-idx)                   Output Shape              Param #
Model3                                   [6, 1]                    --
├─Linear: 1-1                            [6, 3]                    9
├─ReLU: 1-2                              [6, 3]                    --
├─Linear: 1-3                            [6, 1]                    4
├─Sigmoid: 1-4                           [6, 1]                    --
Total params: 13
Trainable params: 13
Non-trainable params: 0
Total mult-adds (M): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

##### Architecture using sequential container

In [62]:
class Model4(nn.Module):
    def __init__(self,features):
        super().__init__()
        self.network =nn.Sequential(
            nn.Linear(features,3),
            nn.ReLU(),
            nn.Linear(3,1),
            nn.Sigmoid()
        )

    def forward(self,x):
        out=self.network(x)
        return out

#### Training

In [68]:
lr=0.01
epochs=10

In [71]:
model4= Model4(X2_norm.shape[1])
optimizer=torch.optim.SGD(model4.parameters(),lr=lr)

In [73]:
for epoch in range(epochs):
    #forward prop

    y_pred=model4(X2_norm)

    #loss

    loss=binary_cross_entropy(y_pred,Y)

    print(f"epoch : {epoch+1} || loss : {loss.item():.4f}")

    # Backpropagation

    loss.backward()


    # updating parameters

    optimizer.step()
    

    # clear the gradients
    optimizer.zero_grad()


epoch : 1 || loss : 0.7017
epoch : 2 || loss : 0.7014
epoch : 3 || loss : 0.7013
epoch : 4 || loss : 0.7012
epoch : 5 || loss : 0.7010
epoch : 6 || loss : 0.7009
epoch : 7 || loss : 0.7008
epoch : 8 || loss : 0.7007
epoch : 9 || loss : 0.7005
epoch : 10 || loss : 0.7004
